In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kacpergregorowicz/house-plant-species")

print("Path to dataset files:", path)

 45%|████▌     | 2.20G/4.85G [01:06<01:51, 25.4MB/s]

In [26]:
from torch.utils.data import Dataset
from PIL import Image
import os
import pandas as pd
from torchvision.models import resnet18
from AircraftDataset import AircraftDataset
from torchvision import transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F

In [27]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [28]:
class AircraftCNN(nn.Module):
    def __init__(self, num_classes):
        super(AircraftCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))  # -> [B, ?, 4, 4]

        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # -> [B, 32, H/2, W/2]
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # -> [B, 64, H/4, W/4]
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # -> [B, 128, H/8, W/8]
        x = self.adaptive_pool(x)  # -> [B, 128, 4, 4]
        x = x.view(x.size(0), -1)  # -> [B, 2048]
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [29]:
def evaluate_model(model, data_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

In [30]:
import torch
from torch import nn, optim
from tqdm import tqdm
import wandb


def train_model(model, train_loader, val_loader, device, epochs=10):
    # wandb.init(project="aircraft-classifier",
    #            config={"epochs": epochs,
    #                    "lr": 1e-4,
    #                    "batch_size": train_loader.batch_size}
    #            )
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    best_acc = 0.0
    patience = 3
    counter = 0

    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}"):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = correct / total
        val_acc = evaluate_model(model, val_loader, device)

        # wandb.log({
        #     "train_loss": running_loss / total,
        #     "train_acc": train_acc,
        #     "val_acc": val_acc,
        #     "epoch": epoch + 1
        # })

        if val_acc > best_acc:
            best_acc = val_acc
            counter = 0
            torch.save(model.state_dict(), "best_model.pth")
            print("✅ Model saved.")
            # wandb.run.summary["best_val_acc"] = best_acc
        else:
            counter += 1
            if counter >= patience:
                print('Early stopping triggered')
                break
        scheduler.step()

        print(f"Epoch {epoch + 1}: Loss={running_loss / total:.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

### Загрузим данные

In [18]:
train_small_dataset = AircraftDataset('data/fgvc-aircraft/train_small.csv', 'data/fgvc-aircraft/images', transform)
val_small_dataset = AircraftDataset('data/fgvc-aircraft/val_small.csv', 'data/fgvc-aircraft/images', transform)

train_small_loader = DataLoader(train_small_dataset, batch_size=16, pin_memory=True, num_workers=4, shuffle=True)
val_small_loader = DataLoader(val_small_dataset, batch_size=16, pin_memory=True, num_workers=4)

In [19]:
model = AircraftCNN(num_classes=100)
model = model.to(device)

In [20]:
train_model(model, train_small_loader, val_small_loader, device, epochs=20)

Epoch 1/20: 100%|██████████| 44/44 [00:11<00:00,  3.68it/s]


✅ Model saved.
Epoch 1: Loss=4.6833, Train Acc=0.0114, Val Acc=0.0143


Epoch 2/20: 100%|██████████| 44/44 [00:11<00:00,  3.95it/s]


✅ Model saved.
Epoch 2: Loss=4.5817, Train Acc=0.0200, Val Acc=0.0200


Epoch 3/20: 100%|██████████| 44/44 [00:10<00:00,  4.04it/s]


...

Epoch 5: Loss=4.4246, Train Acc=0.0343, Val Acc=0.0229


Epoch 6/20: 100%|██████████| 44/44 [00:10<00:00,  4.13it/s]


Epoch 6: Loss=4.3575, Train Acc=0.0514, Val Acc=0.0214


Epoch 7/20: 100%|██████████| 44/44 [00:10<00:00,  4.01it/s]


Early stopping triggered


### Добавим аугментации

In [33]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [34]:
train_small_dataset = AircraftDataset('data/fgvc-aircraft/train_small.csv', 'data/fgvc-aircraft/images', transform)
val_small_dataset = AircraftDataset('data/fgvc-aircraft/val_small.csv', 'data/fgvc-aircraft/images', transform)

train_small_loader = DataLoader(train_small_dataset, batch_size=16, pin_memory=True, num_workers=4, shuffle=True)
val_small_loader = DataLoader(val_small_dataset, batch_size=16, pin_memory=True, num_workers=4)

In [35]:
model = AircraftCNN(num_classes=100)
model = model.to(device)

In [36]:
train_model(model, train_small_loader, val_small_loader, device, epochs=20)

Epoch 1/20: 100%|██████████| 44/44 [00:11<00:00,  3.97it/s]


✅ Model saved.
Epoch 1: Loss=4.6577, Train Acc=0.0129, Val Acc=0.0086


Epoch 2/20: 100%|██████████| 44/44 [00:10<00:00,  4.13it/s]


✅ Model saved.
Epoch 2: Loss=4.5797, Train Acc=0.0186, Val Acc=0.0314


Epoch 3/20: 100%|██████████| 44/44 [00:10<00:00,  4.08it/s]


Epoch 3: Loss=4.5291, Train Acc=0.0300, Val Acc=0.0229


Epoch 4/20: 100%|██████████| 44/44 [00:10<00:00,  4.07it/s]


Epoch 4: Loss=4.4896, Train Acc=0.0171, Val Acc=0.0243


Epoch 5/20: 100%|██████████| 44/44 [00:10<00:00,  4.17it/s]


Early stopping triggered


### Наблюдается огромное недообучение. Ситуация улучшилась с 2.7% до 3.1%, но этого по-прежнему недостаточно. Попробуем изменить архитектуру модели, возьмём предобученный ResNet18 с изменённым последним слоем под 100 классов

In [39]:
model = resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 100)
model = model.to(device)

In [40]:
train_model(model, train_small_loader, val_small_loader, device, epochs=20)

Epoch 1/20: 100%|██████████| 44/44 [00:11<00:00,  4.00it/s]


✅ Model saved.
Epoch 1: Loss=4.7029, Train Acc=0.0271, Val Acc=0.0700


Epoch 2/20: 100%|██████████| 44/44 [00:10<00:00,  4.11it/s]


✅ Model saved.
Epoch 2: Loss=3.9045, Train Acc=0.2271, Val Acc=0.1157


Epoch 3/20: 100%|██████████| 44/44 [00:10<00:00,  4.04it/s]


...

Epoch 11: Loss=0.8004, Train Acc=0.9914, Val Acc=0.3271


Epoch 12/20: 100%|██████████| 44/44 [00:10<00:00,  4.19it/s]


Epoch 12: Loss=0.7324, Train Acc=0.9986, Val Acc=0.3200


Epoch 13/20: 100%|██████████| 44/44 [00:10<00:00,  4.34it/s]


Early stopping triggered


### Модель обучилась в 10 раз лучше, что говорит о том, что изменение архитектуры дало положительные результаты. Недообучение по-прежнему наблюдается, попробуем расширить данные - добавим больше картинок.

In [41]:
train_dataset = AircraftDataset('data/fgvc-aircraft/train.csv', 'data/fgvc-aircraft/images', transform)
val_dataset = AircraftDataset('data/fgvc-aircraft/val.csv', 'data/fgvc-aircraft/images', transform)
test_dataset = AircraftDataset('data/fgvc-aircraft/test.csv', 'data/fgvc-aircraft/images', transform)

train_loader = DataLoader(train_dataset, batch_size=16, pin_memory=True, num_workers=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, pin_memory=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=16, pin_memory=True, num_workers=4)

In [42]:
model = resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 100)
model = model.to(device)

In [43]:
train_model(model, train_loader, val_loader, device, epochs=20)

Epoch 1/20: 100%|██████████| 209/209 [00:36<00:00,  5.66it/s]


✅ Model saved.
Epoch 1: Loss=4.0192, Train Acc=0.1254, Val Acc=0.2667


Epoch 2/20: 100%|██████████| 209/209 [00:35<00:00,  5.88it/s]


✅ Model saved.
Epoch 2: Loss=2.6965, Train Acc=0.4112, Val Acc=0.4356


Epoch 3/20: 100%|██████████| 209/209 [00:35<00:00,  5.87it/s]


...

Epoch 18: Loss=0.0702, Train Acc=0.9988, Val Acc=0.7204


Epoch 19/20: 100%|██████████| 209/209 [00:35<00:00,  5.97it/s]


Epoch 19: Loss=0.0686, Train Acc=0.9991, Val Acc=0.7237


Epoch 20/20: 100%|██████████| 209/209 [00:35<00:00,  5.90it/s]


✅ Model saved.
Epoch 20: Loss=0.0617, Train Acc=0.9994, Val Acc=0.7309


### Расширенный набор данных (Kaggle + Google) улучшил точность модели до приемлемого уровня, цель достигнута. 